<a href="https://colab.research.google.com/github/mkobycheva/recommendation-system/blob/lstm/notebooks/04_lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 - LSTM Model

## 1. Connecting to the repository and Google Drive

In [2]:
!git clone https://github.com/mkobycheva/recommendation-system.git 2>/dev/null \
  || (cd recommendation-system && git pull)

%cd recommendation-system
# !pip install -r requirements.txt -q

import sys
sys.path.insert(0, '.')

from google.colab import drive
drive.mount('/content/drive')

/content/recommendation-system
Mounted at /content/drive


Check out the right branch

In [3]:
!git checkout lstm && git pull

Branch 'lstm' set up to track remote branch 'lstm' from 'origin'.
Switched to a new branch 'lstm'
Already up to date.


## 2. Prepare data

In [4]:
import pandas as pd

DATA_DIR = '/content/drive/MyDrive/recsys-data'

books_train = pd.read_csv(f'{DATA_DIR}/books_train.csv')
books_valid = pd.read_csv(f'{DATA_DIR}/books_valid.csv')
books_test = pd.read_csv(f'{DATA_DIR}/books_test.csv')

movies_train = pd.read_csv(f'{DATA_DIR}/movies_train.csv')
movies_valid = pd.read_csv(f'{DATA_DIR}/movies_valid.csv')
movies_test = pd.read_csv(f'{DATA_DIR}/movies_test.csv')

In [5]:
from src.data.build_matrix import add_domain_item_ids, build_implicit_als_matrix

books_train = add_domain_item_ids(books_train, 'books')
books_valid = add_domain_item_ids(books_valid, 'books')
books_test = add_domain_item_ids(books_test, 'books')

movies_train = add_domain_item_ids(movies_train, 'movies')
movies_valid = add_domain_item_ids(movies_valid, 'movies')
movies_test = add_domain_item_ids(movies_test, 'movies')

train_df = pd.concat([books_train, movies_train], ignore_index=True)
valid_df = pd.concat([books_valid, movies_valid], ignore_index=True)
test_df = pd.concat([books_test, movies_test], ignore_index=True)

train_df[['user_id', 'item_id', 'domain']].head()

,user_id,item_id,domain
0,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0061713244,books
1,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0871139634,books
2,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0385521685,books
3,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0848732634,books
4,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0151014981,books


In [6]:
unique_train_items = train_df['item_id'].unique()

item_to_idx = {item: idx + 1 for idx, item in enumerate(unique_train_items)}

UNK_IDX = len(item_to_idx) + 1

def encode_item_ids(df, mapping, unk_value):
    return df['item_id'].map(mapping).fillna(unk_value).astype(int)

train_df['item_idx'] = encode_item_ids(train_df, item_to_idx, UNK_IDX)
valid_df['item_idx'] = encode_item_ids(valid_df, item_to_idx, UNK_IDX)
test_df['item_idx'] = encode_item_ids(test_df, item_to_idx, UNK_IDX)

NUM_ITEMS = UNK_IDX + 1
print(f"Number of unique items for LSTM (including padding & UNK): {NUM_ITEMS}")

Number of unique items for LSTM (including padding & UNK): 587066


In [7]:
import numpy as np

def generate_train_sequences(df, max_len=10):
    X, y = [], []
    df_sorted = df.sort_values(by=['user_id', 'timestamp'])

    for user_id, group in df_sorted.groupby('user_id'):
        item_list = group['item_idx'].tolist()
        if len(item_list) < 2:
            continue

        for i in range(1, len(item_list)):
            seq = item_list[:i]
            target = item_list[i]

            if len(seq) >= max_len:
                seq = seq[-max_len:]
            else:
                seq = [0] * (max_len - len(seq)) + seq

            X.append(seq)
            y.append(target)

    return np.array(X), np.array(y)


def generate_valid_sequences(df_combined, max_len=10):
    X, y = [], []

    df_sorted = df_combined.sort_values(by=['user_id', 'timestamp'])

    for user_id, group in df_sorted.groupby('user_id'):
        item_list = group['item_idx'].tolist()
        is_val_list = group['is_val'].tolist()

        if len(item_list) < 2:
            continue

        for i in range(1, len(item_list)):
            if is_val_list[i] == 1:
                seq = item_list[:i]
                target = item_list[i]

                if len(seq) >= max_len:
                    seq = seq[-max_len:]
                else:
                    seq = [0] * (max_len - len(seq)) + seq

                X.append(seq)
                y.append(target)

    return np.array(X), np.array(y)


print("Make orders for Train...")
X_train, y_train = generate_train_sequences(train_df, max_len=10)

train_df['is_val'] = 0
valid_df['is_val'] = 1
valid_combined = pd.concat([train_df, valid_df])

print("Make orders for Validation...")
X_val, y_val = generate_valid_sequences(valid_combined, max_len=10)

print(f"Train: X = {X_train.shape}, y = {y_train.shape}")
print(f"Valid: X = {X_val.shape}, y = {y_val.shape}")

Make orders for Train...
Make orders for Validation...
Train: X = (3623625, 10), y = (3623625,)
Valid: X = (254376, 10), y = (254376,)


In [8]:
import torch
from torch.utils.data import Dataset, DataLoader

class RecSysDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = RecSysDataset(X_train, y_train)
val_dataset = RecSysDataset(X_val, y_val)

BATCH_SIZE = 512
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

for sample_x, sample_y in train_loader:
    print(f"Batch inputs (X): {sample_x.shape}")
    print(f"Batch correct outputs (y): {sample_y.shape}")
    break

Batch inputs (X): torch.Size([512, 10])
Batch correct outputs (y): torch.Size([512])


## 3. Main part

In [9]:
import torch.nn as nn

class RecSysLSTM(nn.Module):
    def __init__(self, num_items, embedding_dim=64, hidden_dim=128):
        super(RecSysLSTM, self).__init__()

        self.item_embeddings = nn.Embedding(num_embeddings=num_items,
                                            embedding_dim=embedding_dim,
                                            padding_idx=0)

        self.lstm = nn.LSTM(input_size=embedding_dim,
                            hidden_size=hidden_dim,
                            batch_first=True)

        self.fc = nn.Linear(hidden_dim, num_items)

    def forward(self, x):

        embedded = self.item_embeddings(x)
        out, (hidden, cell) = self.lstm(embedded)
        last_step = out[:, -1, :]
        logits = self.fc(last_step)

        return logits

In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = RecSysLSTM(num_items=NUM_ITEMS, embedding_dim=64, hidden_dim=64).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [11]:
def calculate_map_at_5(logits, targets):
    _, top5_indices = torch.topk(logits, k=5, dim=-1)

    targets_expanded = targets.unsqueeze(-1).expand_as(top5_indices)
    hits = (top5_indices == targets_expanded)

    ranks = torch.arange(1, 6, device=logits.device).float()

    ap_scores = (hits.float() / ranks).sum(dim=-1)

    return ap_scores.mean().item()

In [13]:
!pip install lightning wandb -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 15.1 MB/s eta 0:00:00


In [14]:
import lightning as L
import wandb

class LightningRecSysLSTM(L.LightningModule):
    def __init__(self, num_items, embedding_dim=64, hidden_dim=128, lr=0.001, weight_decay=1e-5):
        super().__init__()
        self.save_hyperparameters()

        self.item_embeddings = nn.Embedding(num_items, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(hidden_dim, num_items)

        self.criterion = nn.CrossEntropyLoss(ignore_index=0)

    def forward(self, x):
        embedded = self.item_embeddings(x)
        out, _ = self.lstm(embedded)
        last_step = self.dropout(out[:, -1, :])
        return self.fc(last_step)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self.forward(x)
        loss = self.criterion(logits, y)

        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self.forward(x)
        loss = self.criterion(logits, y)

        val_map = calculate_map_at_5(logits, y)

        self.log("val_loss", loss, prog_bar=True)
        self.log("val_map_at_5", val_map, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(
            self.parameters(),
            lr=self.hparams.lr,
            weight_decay=self.hparams.weight_decay
        )

In [ ]:
from lightning.pytorch.loggers import WandbLogger

wandb_logger = WandbLogger(project="recsys-amazon-lstm", log_model="all")

drive_model_dir = "/content/drive/MyDrive/recsys-data/saved_models"

trainer = L.Trainer(
    max_epochs=5,
    accelerator="auto",
    logger=wandb_logger,
    enable_checkpointing=True,
    default_root_dir=drive_model_dir
)

lightning_model = LightningRecSysLSTM(num_items=NUM_ITEMS)

wandb_logger.watch(lightning_model, log="all", log_freq=100)

trainer.fit(lightning_model, train_loader, val_loader)

INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/loggers/wandb.py:400: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before ins

┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ item_embeddings │ Embedding        │ 37.6 M │ train │     0 │
│ 1 │ lstm            │ LSTM             │ 99.3 K │ train │     0 │
│ 2 │ dropout         │ Dropout          │      0 │ train │     0 │
│ 3 │ fc              │ Linear           │ 75.7 M │ train │     0 │
│ 4 │ criterion       │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 113 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 113 M                                                                                                
Total estimated model params size (MB): 453.612                                                                    
Modules in train mode: 5                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

In [16]:
from lightning.pytorch.loggers import WandbLogger

wandb_logger = WandbLogger(project="recsys-amazon-lstm", log_model="all")

checkpoint_path = "/content/lightning_logs/version_0/checkpoints/last.ckpt"


lightning_model = LightningRecSysLSTM(num_items=NUM_ITEMS)
trainer = L.Trainer(max_epochs=12, accelerator="auto", logger=wandb_logger)


trainer.fit(lightning_model, train_loader, val_loader, ckpt_path=checkpoint_path)

INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch.utilities

wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: recommender-ai28 (recommender-ai28-kyiv-school-of-economics) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


INFO: Restoring states from the checkpoint path at /content/lightning_logs/version_0/checkpoints/last.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Restoring states from the checkpoint path at /content/lightning_logs/version_0/checkpoints/last.ckpt


FileNotFoundError: Checkpoint file not found: /content/lightning_logs/version_0/checkpoints/last.ckpt

## 4. Evaluate model

In [ ]:
user_history_dict = {}
train_sorted = train_df.sort_values(by=['user_id', 'timestamp'])

for user_id, group in train_sorted.groupby('user_id'):
    item_list = group['item_idx'].tolist()
    if len(item_list) >= 10:
        seq = item_list[-10:]
    else:
        seq = [0] * (10 - len(item_list)) + item_list
    user_history_dict[user_id] = seq

def recommend_for_users(eval_users, target_domain, k=10):
    model.eval()
    recommendations = {}

    mask = np.full(NUM_ITEMS, -np.inf)
    for idx, label in enumerate(encoder.classes_):
        if label.startswith(f"{target_domain}::"):
            mask[idx + 1] = 0.0

    mask_tensor = torch.tensor(mask, dtype=torch.float, device=device)

    batch_size = 512
    for i in range(0, len(eval_users), batch_size):
        batch_users = eval_users[i:i+batch_size]

        X_batch_list = [user_history_dict.get(u, [0] * 10) for u in batch_users]
        X_batch = torch.tensor(X_batch_list, dtype=torch.long, device=device)

        with torch.no_grad():
            logits = model(X_batch)
            masked_logits = logits + mask_tensor.unsqueeze(0)

            _, top_k_idx = torch.topk(masked_logits, k=k, dim=-1)
            top_k_idx = top_k_idx.cpu().numpy()

        for j, u in enumerate(batch_users):
            user_recs_idx = top_k_idx[j]
            user_recs_labels = []
            for idx in user_recs_idx:
                if idx > 0:
                    user_recs_labels.append(encoder.classes_[idx - 1])
                else:
                    user_recs_labels.append("unknown")
            recommendations[u] = user_recs_labels

    return recommendations

In [ ]:
def relevant_items_by_user(split_df, target_domain):
    domain_rows = split_df[split_df['domain'] == target_domain]
    return domain_rows.groupby('user_id')['item_id'].agg(set).to_dict()

In [ ]:
from src.evaluation.metrics import ndcg_at_k, recall_at_k

def evaluate_multi_domain(split_df, k=10):
    """
    Multi-domain: recommend items from BOTH domains simultaneously.
    Model is trained jointly on all domains (collective model).
    Evaluate each domain separately.
    """
    results = {}

    for target_domain in ['books', 'movies']:
        relevant = relevant_items_by_user(split_df, target_domain)
        eval_users = list(relevant.keys())

        print(f"Evaluating {target_domain}: {len(eval_users)} users")

        recs = recommend_for_users(eval_users, target_domain, k=k)

        ndcg_scores, recall_scores = [], []
        for user_id in eval_users:
            user_recs = recs.get(user_id, [])
            user_relevant = relevant.get(user_id, set())
            if not user_relevant:
                continue
            ndcg_scores.append(ndcg_at_k(user_recs, user_relevant, k))
            recall_scores.append(recall_at_k(user_recs, user_relevant, k))

        results[target_domain] = {
            'ndcg@10': round(np.mean(ndcg_scores), 4),
            'recall@10': round(np.mean(recall_scores), 4),
            'n_users': len(ndcg_scores),
        }

    return results


In [ ]:
lightning_model.eval()

K = 10
valid_results = evaluate_multi_domain(valid_df, k=K)

# Виводимо результати
for domain, metrics in valid_results.items():
    print(f"\n{domain}:")
    print(f"  NDCG@10:  {metrics['ndcg_10']}")
    print(f"  Recall@10: {metrics['recall_10']}")

Evaluating books: 127188 users
Evaluating movies: 127188 users

books:
  NDCG@10:   0.0026
  Recall@10: 0.0052
  Users:     127188

movies:
  NDCG@10:   0.0092
  Recall@10: 0.016
  Users:     127188
